<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

Install the core libraries needed for this notebook.

In [ ]:
!pip install pykalman pandas matplotlib yfinance

The openbb_terminal package requires a separate installation process with additional dependencies. Follow the official OpenBB documentation to install it before running this notebook.

## Imports and setup

pandas handles time series data, matplotlib produces the charts, pykalman provides the Kalman filter implementation, and openbb gives us a convenient interface for downloading historical stock prices.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from pykalman import KalmanFilter

## Load historical stock prices

Pull two years of daily closing prices for Lockheed Martin (LMT) to use as our noisy input signal.

In [ ]:
data = yf.download(
    "LMT", start="2013-01-01", end="2015-01-01"
)
prices = data["Close"]

We isolate the "Close" column because the Kalman filter operates on a single series of observations. Working with closing prices is standard practice since they represent the market's consensus value at the end of each trading session.

## Configure and run the Kalman filter

Set up a Kalman filter that models the stock price as a slowly evolving hidden state observed through noisy daily closes.

In [ ]:
kf = KalmanFilter(
    transition_matrices=[1],
    observation_matrices=[1],
    initial_state_mean=0,
    initial_state_covariance=1,
    observation_covariance=1,
    transition_covariance=0.01,
)

The transition matrix of 1 tells the filter that tomorrow's true price is expected to equal today's (a random walk assumption). The key ratio is observation_covariance to transition_covariance. A small transition_covariance (0.01) relative to observation_covariance (1) tells the filter that most tick-to-tick variation is noise, so it should smooth aggressively. Adjusting this ratio is the one knob you have, and it replaces the arbitrary window-length choice entirely.

Run the filter forward through every price in the series, producing a smoothed estimate at each time step.

In [ ]:
state_means, _ = kf.filter(prices.values)
state_means = pd.Series(state_means.flatten(), index=prices.index)

The filter method processes prices sequentially, updating its estimate with each new observation, exactly as it would in a live trading system. We wrap the output in a pandas Series so it aligns with our original date index for easy plotting.

Compute a traditional 30-day simple moving average so we can compare it directly against the Kalman output.

In [ ]:
mean30 = prices.rolling(window=30).mean()

This is the approach the post warns about. The 30-day window is a common default, but changing it to 10 or 60 days would produce a noticeably different line and potentially different trading signals. The Kalman filter avoids that fragility.

## Visualize the full and zoomed comparison

Plot all three series over the full two-year period to see how the Kalman filter tracks price versus the fixed moving average.

In [ ]:
plt.plot(state_means)
plt.plot(prices)
plt.plot(mean30)
plt.title("Kalman filter estimate of average")
plt.legend(["Kalman", "Price", "30-day MA"])
plt.xlabel("Day")
plt.ylabel("Price")
plt.show()

In the full view, notice how the Kalman line hugs the price more closely during steady trends and stays smooth during choppy periods. The 30-day moving average, by contrast, lags behind sharp moves because it weights all 30 days equally regardless of market conditions.

Zoom into the last 200 trading days to see the difference in responsiveness more clearly.

In [ ]:
plt.plot(state_means[-200:])
plt.plot(prices[-200:])
plt.plot(mean30[-200:])
plt.title("Kalman filter estimate of average")
plt.legend(["Kalman", "Price", "30-day MA"])
plt.xlabel("Day")
plt.ylabel("Price")
plt.show()

At this closer scale, the lag of the 30-day moving average becomes obvious around turning points where the price reverses direction. The Kalman filter reacts faster because it reweights new information at every step rather than averaging a fixed block of history. This is the practical payoff described in the post: one fewer arbitrary parameter means one fewer way to overfit your analysis to past data.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.